In [1]:
import yfinance as yf
import pandas as pd
import os

os.environ.pop("http_proxy", None)
os.environ.pop("https_proxy", None)
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)

os.makedirs("data/raw", exist_ok=True)

In [ ]:
import requests
import pandas as pd
import os
import time
from datetime import datetime
from dotenv import load_dotenv
import os

load_dotenv()
TIINGO_API_KEY = os.getenv("TIINGO_API_KEY")

tickers = [
    "AAPL", "MSFT", "GOOGL", "META", "AMZN", "NFLX", "TSLA", "ABNB",
    "NVDA", "AMD", "INTC", "QCOM", "AVGO", "MU", "ARM", "AMAT",
    "V", "MA", "PYPL", "COIN", "HOOD",
    "JPM", "GS", "BAC", "WFC", "MS", "BLK",
    "CRM", "NOW", "SNOW", "PLTR", "NET", "DDOG", "MDB",
    "WMT", "TGT", "COST", "EBAY", "SHOP",
    "UNH", "ISRG", "DXCM",
    "ENPH", "FSLR", "RIVN"
]

os.makedirs("data/raw", exist_ok=True)

end_date = datetime.today().strftime("%Y-%m-%d")
start_date = "2019-01-01"
print(f"下载区间: {start_date} ~ {end_date}")

def download_tiingo(ticker, start, end):
    url = f"https://api.tiingo.com/tiingo/daily/{ticker}/prices"
    params = {
        "startDate": start,
        "endDate": end,
        "token": TIINGO_API_KEY,
        "format": "json"
    }
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json())
    return df

for ticker in tickers:
    success = False
    for attempt in range(3):
        try:
            df = download_tiingo(ticker, start_date, end_date)
            if len(df) > 10:
                # 用复权后的价格（adjClose等），列名统一改成和原来一致的格式
                out = pd.DataFrame({
                    "Date": pd.to_datetime(df["date"]).dt.tz_localize(None),
                    "Open": df["adjOpen"],
                    "High": df["adjHigh"],
                    "Low": df["adjLow"],
                    "Close": df["adjClose"],
                    "Volume": df["adjVolume"]
                })
                out.to_csv(f"data/raw/{ticker}_price.csv", index=False)
                print(f"{ticker} 下载完成，{len(out)} 行")
                success = True
                break
            else:
                print(f"{ticker} 第{attempt+1}次失败（数据过短），重试...")
                time.sleep(3)
        except Exception as e:
            print(f"{ticker} 出错: {e}")
            time.sleep(3)
    if not success:
        print(f"{ticker} 多次重试仍失败")
    time.sleep(1)


下载区间: 2019-01-01 ~ 2026-07-09
AAPL 下载完成，1888 行
MSFT 下载完成，1888 行
GOOGL 下载完成，1888 行
META 下载完成，1888 行
AMZN 下载完成，1888 行
NFLX 下载完成，1888 行
TSLA 下载完成，1888 行
ABNB 下载完成，1398 行
NVDA 下载完成，1888 行
AMD 下载完成，1888 行
INTC 下载完成，1888 行
QCOM 下载完成，1888 行
AVGO 下载完成，1888 行
MU 下载完成，1888 行
ARM 下载完成，705 行
AMAT 下载完成，1888 行
V 下载完成，1888 行
MA 下载完成，1888 行
PYPL 下载完成，1888 行
COIN 下载完成，1314 行
HOOD 下载完成，1240 行
JPM 下载完成，1888 行
GS 下载完成，1888 行
BAC 下载完成，1888 行
WFC 下载完成，1888 行
MS 下载完成，1888 行
BLK 下载完成，1888 行
CRM 下载完成，1888 行
NOW 下载完成，1888 行
SNOW 下载完成，1458 行
PLTR 下载完成，1448 行
NET 下载完成，1712 行
DDOG 下载完成，1708 行
MDB 下载完成，1888 行
WMT 下载完成，1888 行
TGT 下载完成，1888 行
COST 下载完成，1888 行
EBAY 下载完成，1888 行
SHOP 下载完成，1888 行
UNH 下载完成，1888 行
ISRG 下载完成，1888 行
DXCM 下载完成，1888 行
ENPH 下载完成，1888 行
FSLR 下载完成，1888 行
RIVN 下载完成，1167 行


In [3]:
import os

files = [
    f.replace("_price.csv", "")
    for f in os.listdir("data/raw")
    if f.endswith(".csv")
]

print("已下载:", len(files))

missing = sorted(set(tickers) - set(files))

print("缺失:")
print(missing)

已下载: 46
缺失:
[]


In [6]:
import yfinance as yf
import pandas as pd
import time

fundamentals = {}
for ticker in tickers:
    success = False
    for attempt in range(3):
        try:
            stock = yf.Ticker(ticker)
            info = stock.info
            fundamentals[ticker] = {
                "trailingEPS": info.get("trailingEps"),
                "trailingPE": info.get("trailingPE"),
                "totalRevenue": info.get("totalRevenue"),
                "freeCashflow": info.get("freeCashflow"),
                "marketCap": info.get("marketCap"),
            }
            print(f"{ticker} 基本面获取完成")
            success = True
            break
        except Exception as e:
            print(f"{ticker} 第{attempt+1}次失败: {e}")
            time.sleep(15)
    if not success:
        fundamentals[ticker] = {
            "trailingEPS": None, "trailingPE": None,
            "totalRevenue": None, "freeCashflow": None, "marketCap": None
        }
        print(f"{ticker} 多次重试仍失败，记为空值")
    time.sleep(3)

df_fundamentals = pd.DataFrame(fundamentals).T
df_fundamentals.to_csv("data/raw/fundamentals.csv")
df_fundamentals


AAPL 第1次失败: Too Many Requests. Rate limited. Try after a while.


KeyboardInterrupt: 

In [5]:
for ticker in tickers:
    df = pd.read_csv(f"data/raw/{ticker}_price.csv")
    print(f"{ticker}: {len(df)} 行, 列: {list(df.columns)}")

AAPL: 1888 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
MSFT: 1888 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
GOOGL: 1888 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
META: 1888 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
AMZN: 1888 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
NFLX: 1888 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
TSLA: 1888 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
ABNB: 1398 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
NVDA: 1888 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
AMD: 1888 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
INTC: 1888 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
QCOM: 1888 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
AVGO: 1888 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
MU: 1888 行, 列: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
ARM: 705 行, 列: ['Date', 'Open', 'High', 'Low', 'Cl